CELL 1 — INSTALLS, IMPORTS & ENVIRONMENT CHECK

In [ ]:
!pip install -q tensorflow==2.11.0 keras==2.11.0 numpy==1.23.5 pandas==1.5.3 pyarrow==10.0.1 scikit-learn==1.2.2 matplotlib==3.6.3 seaborn==0.12.2 tqdm==4.64.1 h5py==3.8.0

In [ ]:
import os
import gc
import json
import random
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
import tensorflow.keras.backend as K
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

print(f"TF: {tf.__version__}")
print("GPUs:", tf.config.list_physical_devices('GPU'))

CELL 2 — CLASS CFG

In [ ]:
class CFG:
    n_splits = 5
    save_output = True
    output_dir = '/kaggle/working'

    seed = 42
    verbose = 2 #0) silent 1) progress bar 2) one line per epoch

    max_len = 384
    replicas = 8
    lr = 5e-4 * replicas
    weight_decay = 0.1
    lr_min = 1e-6
    epoch = 300 #400
    warmup = 0
    batch_size = 64 * replicas
    snapshot_epochs = []
    swa_epochs = [] #list(range(epoch//2,epoch+1))

    fp16 = True
    fgm = False
    awp = True
    awp_lambda = 0.2
    awp_start_epoch = 15
    dropout_start_epoch = 15
    resume = 0
    decay_type = 'cosine'
    dim = 192
    comment = f'islr-fp16-192-8-seed{seed}'

CELL 3 — DATASET_DF BUILDER

In [ ]:
ROOT = "/kaggle/input/datasets/swaptr/indian-sign-language-mediapipe-holistic-landmarks"
kp_root = os.path.join(ROOT, "keypoints")

with open(f"{ROOT}/label_map.json") as f:
    label_map = json.load(f)

rows = []

for cat in os.listdir(kp_root):
    cat_path = os.path.join(kp_root, cat)
    if not os.path.isdir(cat_path): continue

    for word in os.listdir(cat_path):
        word_path = os.path.join(cat_path, word)
        if not os.path.isdir(word_path): continue

        if word not in label_map:
            print(f"?? missing label: {word}")
            continue

        lbl = label_map[word]

        for f in os.listdir(word_path):
            if f.endswith(".parquet"):
                rows.append({
                    "path": os.path.join(word_path, f),
                    "word": word,
                    "label": lbl
                })

df = pd.DataFrame(rows)
print(df.shape)
df.head()

CELL 4 — TRAIN/VALIDATION SPLIT

In [ ]:
train_df, val_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df["label"],
    random_state=42
)

print(f"Train: {len(train_df)} | Val: {len(val_df)}")
print(f"Classes - Train: {train_df['label'].nunique()}, Val: {val_df['label'].nunique()}")

CELL 5 — CONSTANTS BLOCK

In [ ]:
ROWS_PER_FRAME = 543
MAX_LEN = 384
CROP_LEN = MAX_LEN
NUM_CLASSES  = 263
PAD = -100.
NOSE=[
    1,2,98,327
]
LNOSE = [98]
RNOSE = [327]
LIP = [ 0,
    61, 185, 40, 39, 37, 267, 269, 270, 409,
    291, 146, 91, 181, 84, 17, 314, 405, 321, 375,
    78, 191, 80, 81, 82, 13, 312, 311, 310, 415,
    95, 88, 178, 87, 14, 317, 402, 318, 324, 308,
]
LLIP = [84,181,91,146,61,185,40,39,37,87,178,88,95,78,191,80,81,82]
RLIP = [314,405,321,375,291,409,270,269,267,317,402,318,324,308,415,310,311,312]

POSE = [500, 502, 504, 501, 503, 505, 512, 513]
LPOSE = [513,505,503,501]
RPOSE = [512,504,502,500]

REYE = [
    33, 7, 163, 144, 145, 153, 154, 155, 133,
    246, 161, 160, 159, 158, 157, 173,
]
LEYE = [
    263, 249, 390, 373, 374, 380, 381, 382, 362,
    466, 388, 387, 386, 385, 384, 398,
]

LHAND = np.arange(468, 489).tolist()
RHAND = np.arange(522, 543).tolist()

POINT_LANDMARKS = LIP + LHAND + RHAND + NOSE + REYE + LEYE #+POSE

NUM_NODES = len(POINT_LANDMARKS)
CHANNELS = 6*NUM_NODES

print(NUM_NODES)
print(CHANNELS)

def interp1d_(x, target_len, method='random'):
    length = tf.shape(x)[1]
    target_len = tf.maximum(1,target_len)
    if method == 'random':
        if tf.random.uniform(()) < 0.33:
            x = tf.image.resize(x, (target_len,tf.shape(x)[1]),'bilinear')
        else:
            if tf.random.uniform(()) < 0.5:
                x = tf.image.resize(x, (target_len,tf.shape(x)[1]),'bicubic')
            else:
                x = tf.image.resize(x, (target_len,tf.shape(x)[1]),'nearest')
    else:
        x = tf.image.resize(x, (target_len,tf.shape(x)[1]),method)
    return x

def tf_nan_mean(x, axis=0, keepdims=False):
    return tf.reduce_sum(tf.where(tf.math.is_nan(x), tf.zeros_like(x), x), axis=axis, keepdims=keepdims) / tf.reduce_sum(tf.where(tf.math.is_nan(x), tf.zeros_like(x), tf.ones_like(x)), axis=axis, keepdims=keepdims)

def tf_nan_std(x, center=None, axis=0, keepdims=False):
    if center is None:
        center = tf_nan_mean(x, axis=axis,  keepdims=True)
    d = x - center
    return tf.math.sqrt(tf_nan_mean(d * d, axis=axis, keepdims=keepdims))

class Preprocess(tf.keras.layers.Layer):
    def __init__(self, max_len=MAX_LEN, point_landmarks=POINT_LANDMARKS, **kwargs):
        super().__init__(**kwargs)
        self.max_len = max_len
        self.point_landmarks = point_landmarks

    def call(self, inputs):
        if tf.rank(inputs) == 3:
            x = inputs[None,...]
        else:
            x = inputs

        mean = tf_nan_mean(tf.gather(x, [17], axis=2), axis=[1,2], keepdims=True)
        mean = tf.where(tf.math.is_nan(mean), tf.constant(0.5,x.dtype), mean)
        x = tf.gather(x, self.point_landmarks, axis=2) #N,T,P,C
        std = tf_nan_std(x, center=mean, axis=[1,2], keepdims=True)

        x = (x - mean)/std

        if self.max_len is not None:
            x = x[:,:self.max_len]
        length = tf.shape(x)[1]
        x = x[...,:2]

        dx = tf.cond(tf.shape(x)[1]>1,lambda:tf.pad(x[:,1:] - x[:,:-1], [[0,0],[0,1],[0,0],[0,0]]),lambda:tf.zeros_like(x))

        dx2 = tf.cond(tf.shape(x)[1]>2,lambda:tf.pad(x[:,2:] - x[:,:-2], [[0,0],[0,2],[0,0],[0,0]]),lambda:tf.zeros_like(x))

        x = tf.concat([
            tf.reshape(x, (-1,length,2*len(self.point_landmarks))),
            tf.reshape(dx, (-1,length,2*len(self.point_landmarks))),
            tf.reshape(dx2, (-1,length,2*len(self.point_landmarks))),
        ], axis = -1)

        x = tf.where(tf.math.is_nan(x),tf.constant(0.,x.dtype),x)

        return x

CELL 8 — MODEL BLOCK CLASSES

In [ ]:
class ECA(tf.keras.layers.Layer):
    def __init__(self, kernel_size=5, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True
        self.kernel_size = kernel_size
        self.conv = tf.keras.layers.Conv1D(1, kernel_size=kernel_size, strides=1, padding="same", use_bias=False)

    def call(self, inputs, mask=None):
        nn = tf.keras.layers.GlobalAveragePooling1D()(inputs, mask=mask)
        nn = tf.expand_dims(nn, -1)
        nn = self.conv(nn)
        nn = tf.squeeze(nn, -1)
        nn = tf.nn.sigmoid(nn)
        nn = nn[:,None,:]
        return inputs * nn

class LateDropout(tf.keras.layers.Layer):
    def __init__(self, rate, noise_shape=None, start_step=0, **kwargs):
        super().__init__(**kwargs)
        self.supports_masking = True
        self.rate = rate
        self.start_step = start_step
        self.dropout = tf.keras.layers.Dropout(rate, noise_shape=noise_shape)

    def build(self, input_shape):
        super().build(input_shape)
        agg = tf.VariableAggregation.ONLY_FIRST_REPLICA
        self._train_counter = tf.Variable(0, dtype="int64", aggregation=agg, trainable=False)

    def call(self, inputs, training=False):
        x = tf.cond(self._train_counter < self.start_step, lambda:inputs, lambda:self.dropout(inputs, training=training))
        if training:
            self._train_counter.assign_add(1)
        return x

class CausalDWConv1D(tf.keras.layers.Layer):
    def __init__(self,
        kernel_size=17,
        dilation_rate=1,
        use_bias=False,
        depthwise_initializer='glorot_uniform',
        name='', **kwargs):
        super().__init__(name=name,**kwargs)
        self.causal_pad = tf.keras.layers.ZeroPadding1D((dilation_rate*(kernel_size-1),0),name=name + '_pad')
        self.dw_conv = tf.keras.layers.DepthwiseConv1D(
                            kernel_size,
                            strides=1,
                            dilation_rate=dilation_rate,
                            padding='valid',
                            use_bias=use_bias,
                            depthwise_initializer=depthwise_initializer,
                            name=name + '_dwconv')
        self.supports_masking = True

    def call(self, inputs):
        x = self.causal_pad(inputs)
        x = self.dw_conv(x)
        return x

def Conv1DBlock(channel_size,
          kernel_size,
          dilation_rate=1,
          drop_rate=0.0,
          expand_ratio=2,
          se_ratio=0.25,
          activation='swish',
          name=None):
    '''
    efficient conv1d block, @hoyso48
    '''
    if name is None:
        name = str(tf.keras.backend.get_uid("mbblock"))
    # Expansion phase
    def apply(inputs):
        channels_in = tf.keras.backend.int_shape(inputs)[-1]
        channels_expand = channels_in * expand_ratio

        skip = inputs

        x = tf.keras.layers.Dense(
            channels_expand,
            use_bias=True,
            activation=activation,
            name=name + '_expand_conv')(inputs)

        # Depthwise Convolution
        x = CausalDWConv1D(kernel_size,
            dilation_rate=dilation_rate,
            use_bias=False,
            name=name + '_dwconv')(x)

        x = tf.keras.layers.BatchNormalization(momentum=0.95, name=name + '_bn')(x)

        x  = ECA()(x)

        x = tf.keras.layers.Dense(
            channel_size,
            use_bias=True,
            name=name + '_project_conv')(x)

        if drop_rate > 0:
            x = tf.keras.layers.Dropout(drop_rate, noise_shape=(None,1,1), name=name + '_drop')(x)

        if (channels_in == channel_size):
            x = tf.keras.layers.add([x, skip], name=name + '_add')
        return x

    return apply

In [ ]:
class MultiHeadSelfAttention(tf.keras.layers.Layer):
    def __init__(self, dim=256, num_heads=4, dropout=0, **kwargs):
        super().__init__(**kwargs)
        self.dim = dim
        self.scale = self.dim ** -0.5
        self.num_heads = num_heads
        self.qkv = tf.keras.layers.Dense(3 * dim, use_bias=False)
        self.drop1 = tf.keras.layers.Dropout(dropout)
        self.proj = tf.keras.layers.Dense(dim, use_bias=False)
        self.supports_masking = True

    def call(self, inputs, mask=None):
        qkv = self.qkv(inputs)
        qkv = tf.keras.layers.Permute((2, 1, 3))(tf.keras.layers.Reshape((-1, self.num_heads, self.dim * 3 // self.num_heads))(qkv))
        q, k, v = tf.split(qkv, [self.dim // self.num_heads] * 3, axis=-1)

        attn = tf.matmul(q, k, transpose_b=True) * self.scale

        if mask is not None:
            mask = mask[:, None, None, :]

        attn = tf.keras.layers.Softmax(axis=-1)(attn, mask=mask)
        attn = self.drop1(attn)

        x = attn @ v
        x = tf.keras.layers.Reshape((-1, self.dim))(tf.keras.layers.Permute((2, 1, 3))(x))
        x = self.proj(x)
        return x


def TransformerBlock(dim=256, num_heads=4, expand=4, attn_dropout=0.2, drop_rate=0.2, activation='swish'):
    def apply(inputs):
        x = inputs
        x = tf.keras.layers.BatchNormalization(momentum=0.95)(x)
        x = MultiHeadSelfAttention(dim=dim,num_heads=num_heads,dropout=attn_dropout)(x)
        x = tf.keras.layers.Dropout(drop_rate, noise_shape=(None,1,1))(x)
        x = tf.keras.layers.Add()([inputs, x])
        attn_out = x

        x = tf.keras.layers.BatchNormalization(momentum=0.95)(x)
        x = tf.keras.layers.Dense(dim*expand, use_bias=False, activation=activation)(x)
        x = tf.keras.layers.Dense(dim, use_bias=False)(x)
        x = tf.keras.layers.Dropout(drop_rate, noise_shape=(None,1,1))(x)
        x = tf.keras.layers.Add()([attn_out, x])
        return x
    return apply

CELL 9 — GET_MODEL DEFINITION

In [ ]:
def get_model(max_len=64, dropout_step=0, dim=192):
    inp = tf.keras.Input((max_len,CHANNELS))
    x = tf.keras.layers.Masking(mask_value=PAD,input_shape=(max_len,CHANNELS))(inp)
    ksize = 17
    x = tf.keras.layers.Dense(dim, use_bias=False,name='stem_conv')(x)
    x = tf.keras.layers.BatchNormalization(momentum=0.95,name='stem_bn')(x)

    x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
    x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
    x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
    x = TransformerBlock(dim,expand=2)(x)

    x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
    x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
    x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
    x = TransformerBlock(dim,expand=2)(x)

    if dim == 384: #for the 4x sized model
        x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
        x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
        x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
        x = TransformerBlock(dim,expand=2)(x)

        x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
        x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
        x = Conv1DBlock(dim,ksize,drop_rate=0.2)(x)
        x = TransformerBlock(dim,expand=2)(x)

    x = tf.keras.layers.Dense(dim*2,activation=None,name='top_conv')(x)
    x = tf.keras.layers.GlobalAveragePooling1D()(x)
    x = LateDropout(0.8, start_step=dropout_step)(x)
    x = tf.keras.layers.Dense(NUM_CLASSES,name='classifier')(x)
    return tf.keras.Model(inp, x)

CELL 10 — LOAD_PARQUET_FILE DEFINITION

In [ ]:
def load_parquet_file(path):
    df = pd.read_parquet(path)
    frames = sorted(df["frame"].unique())

    f_idx = {f: i for i, f in enumerate(frames)}
    landmarks = np.full((len(frames), 543, 3), np.nan, dtype=np.float32)

    offsets = {
        "face": 0,
        "left_hand": 468,
        "pose": 489,
        "right_hand": 522
    }

    for row in df.itertuples():
        if row.type not in offsets: continue
        if row.type == "face" and row.landmark_index > 467: continue

        idx = offsets[row.type] + row.landmark_index
        f = f_idx[row.frame]

        landmarks[f, idx] = [row.x, row.y, row.z]

    return landmarks

CELL 11 — PREPROCESS INSTANCE CREATION

In [ ]:
PREPROCESS = Preprocess()
print("Preprocess layer created")

CELL 12 — ASL MODEL BUILD + WEIGHT LOADING

In [ ]:
OLD_NUM_CLASSES = NUM_CLASSES
NUM_CLASSES = 250
asl_model = get_model(max_len=384, dim=192)
print("ASL model built")

asl_model.load_weights("ASL_MODEL_WEIGHT_PATH_HERE") # Upload the model weights as Input Dataset, and paste the path here
print("Winner weights loaded")

print(asl_model.get_layer("classifier").weights[0].shape)
NUM_CLASSES = 263
print(NUM_CLASSES)

CELL 13 — ISL MODEL BUILD + BACKBONE TRANSFER

In [ ]:
NUM_CLASSES = 263
isl_model = get_model(max_len=384, dim=192)
print("model built.")

# transfer weights from asl backbone
for layer in isl_model.layers:
    if layer.name == "classifier":
        continue
    try:
        layer.set_weights(asl_model.get_layer(layer.name).get_weights())
    except Exception:
        pass

print("weights transferred.")

CELL 14 — PROCESS_SAMPLE DEFINITION

In [ ]:
def process_sample(path, label):
    landmarks = load_parquet_file(path)
    tensor = tf.convert_to_tensor(landmarks, dtype=tf.float32)

    x = PREPROCESS(tensor)[0].numpy()
    seq_len = x.shape[0]

    if seq_len > MAX_LEN:
        x = x[:MAX_LEN]
    elif seq_len < MAX_LEN:
        pad_len = MAX_LEN - seq_len
        x = np.pad(x, ((0, pad_len), (0, 0)), constant_values=PAD)

    y = np.zeros(NUM_CLASSES, dtype=np.float32)
    y[label] = 1.0

    return x.astype(np.float32), y

CELL 15 — X_TRAIN GENERATION

In [ ]:
X_train = np.zeros((len(train_df), MAX_LEN, CHANNELS), dtype=np.float32)
y_train = np.zeros((len(train_df), NUM_CLASSES), dtype=np.float32)

for i, (_, row) in enumerate(train_df.iterrows()):
    x, y = process_sample(row["path"], row["label"])
    X_train[i] = x
    y_train[i] = y

    if i % 100 == 0:
        print(f"{i}/{len(train_df)}")

print("train done.")

CELL 16 — X_VALID GENERATION

In [ ]:
X_val = np.zeros((len(val_df), MAX_LEN, CHANNELS), dtype=np.float32)
y_val = np.zeros((len(val_df), NUM_CLASSES), dtype=np.float32)

for i, (_, row) in enumerate(val_df.iterrows()):
    x, y = process_sample(row["path"], row["label"])
    X_val[i] = x
    y_val[i] = y

    if i % 100 == 0:
        print(f"{i}/{len(val_df)}")

print("val done.")

CELL 17 — FINAL DATASET CREATION (tf.data from arrays)

In [ ]:
train_ds = tf.data.Dataset.from_tensor_slices((X_train, y_train))
train_ds = train_ds.shuffle(2048).batch(32).prefetch(tf.data.AUTOTUNE)

val_ds = tf.data.Dataset.from_tensor_slices((X_val, y_val))
val_ds = val_ds.batch(32).prefetch(tf.data.AUTOTUNE)

for x, y in train_ds.take(1):
    print(f"Train X: {x.shape} | Y: {y.shape}")

for x, y in val_ds.take(1):
    print(f"Val X: {x.shape} | Y: {y.shape}")

CELL 18 — PRE-TRAINING MODEL SMOKE TEST

In [ ]:
for x, y in train_ds.take(1):
    pred = isl_model(x)
    print(pred.shape)
    break

CELL 19 — MEMORY MANAGEMENT

In [ ]:
import gc
gc.collect()
print("Train GB:", X_train.nbytes / 1024**3)
print("Val GB:", X_val.nbytes / 1024**3)  #  Fixed name

CELL 20 — PARAM COUNT

In [ ]:
print(isl_model.count_params())

CELL 21 — COMPILE

In [ ]:
isl_model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-4),
    loss=tf.keras.losses.CategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)
print("compiled.")

CELL 22 — TRAINING

In [ ]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping, ReduceLROnPlateau
checkpoint = ModelCheckpoint(
    "best_include_model.h5",
    monitor="val_accuracy",
    mode="max",
    save_best_only=True,
    save_weights_only=True,
    verbose=1
)
early_stop = EarlyStopping(
    monitor="val_loss",
    patience=8,
    restore_best_weights=True,
    verbose=1
)
reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=3,
    verbose=1,
    min_lr=1e-6
)


In [ ]:
history = isl_model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=60,
    callbacks=[
        checkpoint,
        early_stop,
        reduce_lr
    ]
)

CELL 23 — SAVE

In [ ]:
isl_model.save_weights("include_signvox_epoch60_manual.h5")
print("saved weights manual.")

In [ ]:
isl_model.save("include_aigvox_epoch60_full.h5")
print("saved full model.")

In [ ]:
best_path = "best_include_model.h5"
print("Exists:", os.path.exists(best_path))
print("Size (MB):", os.path.getsize(best_path)/1024/1024)